# 🚲 Outliers y limpieza: el taller de VeloCity — **SOLUCIÓN**

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/soluciones/07_outliers_wrangling_solucion.ipynb)

**Objetivos:** detectar duplicados, normalizar texto sucio, cazar outliers con IQR, discretizar con `pd.cut` y codificar con `map` — el kit completo de limpieza.

🔎 **Apóyate en el visualizador:** [Outliers](https://mati3939.github.io/visualizador-numpy-pandas/#outliers) · [Data wrangling](https://mati3939.github.io/visualizador-numpy-pandas/#wrangling) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: VeloCity, arriendo de bicicletas

El sistema de arriendo VeloCity exporta sus viajes desde tres apps distintas y el resultado es un desastre: viajes duplicados, duraciones imposibles y la columna de tipo de usuario escrita de cuatro maneras — el mismo caos del **Control 4** del curso. Te toca dejarlo utilizable.

## 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(404)
n = 60
viajes = pd.DataFrame({
    'id_viaje': np.arange(1, n + 1),
    'duracion_min': rng.normal(25, 8, n).round(1).clip(4, None),
    'tipo_usuario': rng.choice(['MENSUAL', 'mensual ', 'Mensual', 'diario', 'DIARIO'], n),
    'comuna': rng.choice(['Concepción', 'Talcahuano', 'San Pedro'], n),
})
# 3 viajes con duraciones absurdas: bicicletas mal ancladas que siguieron 'en viaje'
viajes.loc[[7, 31, 52], 'duracion_min'] = [720.0, 545.0, 610.0]
# la exportación duplicó algunas filas completas
viajes = pd.concat([viajes, viajes.iloc[[3, 15, 28]]], ignore_index=False)
viajes = viajes.sample(frac=1, random_state=1).reset_index(drop=True)
print(len(viajes), 'filas')
viajes.head()

## 1 · Calentamiento

Las cuatro operaciones de hoy están animadas en [https://mati3939.github.io/visualizador-numpy-pandas/#wrangling](https://mati3939.github.io/visualizador-numpy-pandas/#wrangling) y [https://mati3939.github.io/visualizador-numpy-pandas/#outliers](https://mati3939.github.io/visualizador-numpy-pandas/#outliers).

### 1.1 ¿Cuántas filas son clones? ⭐

Cuenta en `n_dup` cuántas filas están **duplicadas exactas** (con `duplicated()`), y construye `viajes_unicos` sin los clones. Verifica el largo resultante.

<details><summary>💡 Pista (haz clic)</summary>

`duplicated()` marca como duplicado desde la SEGUNDA aparición en adelante — la original no cuenta.

</details>

In [ ]:
n_dup = viajes.duplicated().sum()          # keep='first': la primera copia no se marca
viajes_unicos = viajes.drop_duplicates()

print(n_dup, 'duplicados ·', len(viajes_unicos), 'filas únicas')

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert n_dup == viajes.duplicated().sum()
assert len(viajes_unicos) == len(viajes) - n_dup
assert viajes_unicos.duplicated().sum() == 0, 'no debe quedar ningún duplicado'
print('✅ ¡Correcto!')

### 1.2 Un solo formato de usuario ⭐

Mira `viajes_unicos['tipo_usuario'].unique()`: el mismo concepto está escrito de varias formas (mayúsculas, espacios colgando). Normaliza la columna en `tipo_limpio` con `.str.strip().str.lower()` — deben quedar solo `'mensual'` y `'diario'`.

<details><summary>💡 Pista (haz clic)</summary>

Los métodos de texto van tras el accesor `.str` — y se pueden encadenar: `.str.strip().str.lower()`.

</details>

In [ ]:
print(viajes_unicos['tipo_usuario'].unique())

tipo_limpio = viajes_unicos['tipo_usuario'].str.strip().str.lower()
# strip saca los espacios de los bordes; lower unifica mayúsculas

print(tipo_limpio.unique())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert set(tipo_limpio.unique()) == {'mensual', 'diario'}
assert len(tipo_limpio) == len(viajes_unicos)
print('✅ ¡Correcto!')

## 2 · Núcleo

### 2.1 Cazar outliers con IQR ⭐⭐

Sobre `viajes_unicos['duracion_min']` calcula `Q1`, `Q3`, `IQR` y los límites `lim_inf` / `lim_sup` con el factor clásico 1.5. Filtra en `outliers` las filas fuera de los límites. ¿Aparecen las bicicletas mal ancladas?

<details><summary>💡 Pista (haz clic)</summary>

Es la misma cuenta del boxplot interactivo: https://mati3939.github.io/visualizador-numpy-pandas/#outliers — ahí puedes mover el umbral y ver qué puntos caen.

</details>

In [ ]:
Q1, Q3 = viajes_unicos['duracion_min'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = viajes_unicos[(viajes_unicos['duracion_min'] < lim_inf) |
                         (viajes_unicos['duracion_min'] > lim_sup)]

print(f'límites [{lim_inf:.1f}, {lim_sup:.1f}]')
print(outliers[['id_viaje', 'duracion_min']])

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert IQR == Q3 - Q1
assert ((outliers['duracion_min'] < lim_inf) | (outliers['duracion_min'] > lim_sup)).all()
assert set([720.0, 545.0, 610.0]).issubset(set(outliers['duracion_min'])), 'las 3 bicicletas mal ancladas deben caer como outliers'
print('✅ ¡Correcto!')

### 2.2 ¿Cuánto mienten los outliers? ⭐⭐

Calcula media y mediana de `duracion_min` **con** outliers (`media_con`, `mediana_con`) y **sin** ellos (`media_sin`, `mediana_sin`, usando los límites del ejercicio anterior). En el comentario responde: ¿cuál de las dos métricas se movió más y por qué?

In [ ]:
media_con = viajes_unicos['duracion_min'].mean()
mediana_con = viajes_unicos['duracion_min'].median()
sin_out = viajes_unicos[(viajes_unicos['duracion_min'] >= lim_inf) &
                        (viajes_unicos['duracion_min'] <= lim_sup)]
media_sin = sin_out['duracion_min'].mean()
mediana_sin = sin_out['duracion_min'].median()

# se movió más la MEDIA: los outliers la arrastran hacia arriba porque suma
# todos los valores; la mediana solo mira el del medio y apenas se inmuta
print(f'media {media_con:.1f} -> {media_sin:.1f}')
print(f'mediana {mediana_con:.1f} -> {mediana_sin:.1f}')

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(sin_out) == len(viajes_unicos) - len(outliers)
assert media_con > media_sin, 'los outliers gigantes deben inflar la media'
assert abs(media_con - media_sin) > abs(mediana_con - mediana_sin), 'la media debe moverse más que la mediana'
print('✅ ¡Correcto!')

### 2.3 Tramos con pd.cut (y su NaN silencioso) ⭐⭐

Discretiza la duración de `sin_out` en `tramo`: bins `[0, 15, 30, np.inf]` con etiquetas `['corto', 'medio', 'largo']`. ¿Por qué el último bin es `np.inf` y no un número «suficientemente grande»? Escríbelo en el comentario.

<details><summary>💡 Pista (haz clic)</summary>

Prueba mentalmente un viaje de 31 minutos con bins [0, 15, 30]: no cae en ninguno → NaN silencioso. El detective de bugs del visualizador tiene este caso exacto.

</details>

In [ ]:
tramo = pd.cut(sin_out['duracion_min'], bins=[0, 15, 30, np.inf],
               labels=['corto', 'medio', 'largo'])

# el último bin es np.inf porque si un valor queda FUERA de los bins,
# pd.cut lo convierte en NaN sin avisar — np.inf garantiza que nada se escape
print(tramo.value_counts())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert tramo.isnull().sum() == 0, 'ningún viaje debe quedar fuera de los tramos'
assert set(tramo.unique()).issubset({'corto', 'medio', 'largo'})
assert tramo.value_counts().sum() == len(sin_out)
print('✅ ¡Correcto!')

### 2.4 map y su trampa ⭐⭐

Codifica `tipo_limpio` a números en `tipo_cod`: `'mensual' → 1`, `'diario' → 0`, usando `.map({...})`. Después responde con un assert propio: ¿qué habría pasado si el diccionario solo trajera `'mensual'`?

<details><summary>💡 Pista (haz clic)</summary>

`map` no perdona: clave ausente = NaN. Si solo quieres reemplazar ALGUNOS valores y dejar el resto intacto, eso es `replace`.

</details>

In [ ]:
tipo_cod = tipo_limpio.map({'mensual': 1, 'diario': 0})

# experimento: map con el diccionario incompleto
experimento = tipo_limpio.map({'mensual': 1})
# todo lo que no está en el diccionario se vuelve NaN: los 'diario' se pierden
print(tipo_cod.value_counts())
print('NaN en el experimento:', experimento.isnull().sum())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert set(tipo_cod.unique()) == {0, 1}
assert tipo_cod.isnull().sum() == 0
assert experimento.isnull().sum() == (tipo_limpio == 'diario').sum(), 'con el dict incompleto, cada diario debió volverse NaN'
print('✅ ¡Correcto!')

## 3 · Desafío

### 3.1 El pipeline completo 🏁 ⭐⭐⭐

Reconstruye todo el proceso en una sola pasada sobre `viajes` (el original sucio) y guarda el resultado en `limpio`: ① sin duplicados exactos, ② `tipo_usuario` normalizado, ③ sin outliers de duración (IQR 1.5 calculado sobre los datos ya deduplicados), ④ con la columna nueva `tramo`. Reporta `resumen`: viajes por tramo y tipo de usuario (groupby doble o pivot, como prefieras).

<details><summary>💡 Pista (haz clic)</summary>

El orden importa: deduplicar ANTES de calcular los cuartiles (los clones distorsionan los percentiles) y discretizar al final, sobre los datos ya sanos.

</details>

In [ ]:
limpio = viajes.drop_duplicates().copy()
limpio['tipo_usuario'] = limpio['tipo_usuario'].str.strip().str.lower()
_q1, _q3 = limpio['duracion_min'].quantile([0.25, 0.75])
_iqr = _q3 - _q1
limpio = limpio[(limpio['duracion_min'] >= _q1 - 1.5 * _iqr) &
                (limpio['duracion_min'] <= _q3 + 1.5 * _iqr)]
limpio['tramo'] = pd.cut(limpio['duracion_min'], bins=[0, 15, 30, np.inf],
                         labels=['corto', 'medio', 'largo'])
resumen = limpio.groupby(['tramo', 'tipo_usuario'], observed=True).size()

print(len(viajes), '->', len(limpio))
print(resumen)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert limpio.duplicated().sum() == 0
assert set(limpio['tipo_usuario'].unique()) == {'mensual', 'diario'}
assert limpio['duracion_min'].max() < 500, 'las bicicletas mal ancladas deben quedar fuera'
assert 'tramo' in limpio.columns and limpio['tramo'].isnull().sum() == 0
assert resumen.sum() == len(limpio)
print('✅ ¡Correcto!')

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **Wrangling**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).